# Single Topic RAG Evaluation Notebook

This notebook evaluates the RAG system's performance on **single-document question answering**.

## Evaluation Methodology

The evaluation tests three distinct question types:

1. **Single Passage Questions** - Answers can be found in a single document chunk (tests basic retrieval and comprehension)
2. **Multi Passage Questions** - Answers require synthesizing information from multiple chunks (tests information aggregation)
3. **No Answer Questions** - Questions that cannot be answered from the document (tests hallucination prevention)

## Scoring

Uses an **LLM-as-judge** approach with 0-5 scoring scale:
- Questions are evaluated by comparing generated answers against reference answers
- Provides both quantitative scores and qualitative feedback

## Dataset

- 20 diverse documents from various domains (gaming wikis, technical docs, academic papers, fiction, etc.)
- Document sizes range from 2 to 117 pages
- Each document is chunked and stored in isolated vector store collections

In [2]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv, find_dotenv

from doc_chat.evaluation.rag_judge_agent import evaluate_response
from doc_chat.llm import create_llm
from doc_chat.rag.citation_retrieval_chain import CitationRetrievalChain
from doc_chat.rag.document_loader import DocumentLoader
from doc_chat.rag.multi_tenant_vector_store import MultiTenantVectorStore

_ = load_dotenv(find_dotenv())

## 1. Load Document Metadata

Load the CSV file containing metadata for all 20 evaluation documents including:


In [3]:
input_dir = '../../data/single_topic_rag_evaluation_dataset/processed'
documents_file = os.path.join(input_dir, 'documents_processed_2.csv')
pd_documents = pd.read_csv(documents_file)
pd_documents

,index,source_url,text,num_pages,total_word_count,avg_words_per_page,total_char_count,avg_chars_per_page
0,0,https://enterthegungeon.fandom.com/wiki/Bullet...,Bullet Kin\nBullet Kin are one of the most com...,7,1847.0,263.0,10609.0,1515.0
1,1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,---The Paths through the Underground/Underdark...,2,470.0,235.0,2737.0,1368.0
2,2,https://bytes-and-nibbles.web.app/bytes/stici-...,Semantic and Textual Inference Chatbot Interfa...,11,3786.0,344.0,22113.0,2010.0
3,3,https://github.com/llmware-ai/llmware,llmware\n\nBuilding Enterprise RAG Pipelines w...,13,3007.0,231.0,20333.0,1564.0
4,4,https://docs.marimo.io/recipes.html,Recipes\nThis page includes code snippets or “...,11,1797.0,163.0,11257.0,1023.0
5,5,https://towardsdatascience.com/how-to-maximize...,How to Maximize Your Impact as a Data Scientis...,8,2446.0,305.0,14217.0,1777.0
6,6,https://ec.europa.eu/commission/presscorner/de...,Why do we need to regulate the use of Artifici...,18,5399.0,299.0,35372.0,1965.0
7,7,https://bg3.wiki/wiki/The_Emperor,The Emperor is a mind flayer who appears in Ba...,10,2973.0,297.0,17889.0,1788.0
8,8,https://whattocook.substack.com/p/so-into-nort...,so into northern spain!\nour magical urban-plu...,6,2046.0,341.0,11259.0,1876.0
9,9,https://dmtalkies.com/the-zone-of-interest-end...,‘The Zone Of Interest’ Ending Explained & Film...,6,2325.0,387.0,13467.0,2244.0


## 2. Load and Split Documents

Process each PDF document using the `DocumentLoader`:
- Loads the full document text
- Splits into chunks using the default chunking strategy
- Records the number of splits per document for analysis

In [4]:
all_splits = []
for index, row in pd_documents.iterrows():
    doc_path = f"{input_dir}/pdf_2/document_{index}.pdf"
    loader = DocumentLoader(doc_path)
    documents, splits = loader.load_and_split()
    pd_documents.loc[index, 'num_splits'] = int(len(splits))
    all_splits.append(splits)

pd_documents['num_splits'] = pd_documents['num_splits'].astype(int)
pd_documents

,index,source_url,text,num_pages,total_word_count,avg_words_per_page,total_char_count,avg_chars_per_page,num_splits
0,0,https://enterthegungeon.fandom.com/wiki/Bullet...,Bullet Kin\nBullet Kin are one of the most com...,7,1847.0,263.0,10609.0,1515.0,14
1,1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,---The Paths through the Underground/Underdark...,2,470.0,235.0,2737.0,1368.0,4
2,2,https://bytes-and-nibbles.web.app/bytes/stici-...,Semantic and Textual Inference Chatbot Interfa...,11,3786.0,344.0,22113.0,2010.0,30
3,3,https://github.com/llmware-ai/llmware,llmware\n\nBuilding Enterprise RAG Pipelines w...,13,3007.0,231.0,20333.0,1564.0,27
4,4,https://docs.marimo.io/recipes.html,Recipes\nThis page includes code snippets or “...,11,1797.0,163.0,11257.0,1023.0,17
5,5,https://towardsdatascience.com/how-to-maximize...,How to Maximize Your Impact as a Data Scientis...,8,2446.0,305.0,14217.0,1777.0,18
6,6,https://ec.europa.eu/commission/presscorner/de...,Why do we need to regulate the use of Artifici...,18,5399.0,299.0,35372.0,1965.0,52
7,7,https://bg3.wiki/wiki/The_Emperor,The Emperor is a mind flayer who appears in Ba...,10,2973.0,297.0,17889.0,1788.0,25
8,8,https://whattocook.substack.com/p/so-into-nort...,so into northern spain!\nour magical urban-plu...,6,2046.0,341.0,11259.0,1876.0,17
9,9,https://dmtalkies.com/the-zone-of-interest-end...,‘The Zone Of Interest’ Ending Explained & Film...,6,2325.0,387.0,13467.0,2244.0,16


## 3. Initialize Vector Store

Create a `MultiTenantVectorStore` with:
- **Backend**: ChromaDB for persistent storage
- **Embedding Model**: OpenAI's `text-embedding-3-small`
- **Architecture**: Multi-tenant design allowing separate collections per document

In [ ]:
chroma_persist_directory = '/Users/patrick/projects/doc-chat/app_data/chroma_eval'
if not os.path.exists(chroma_persist_directory):
    os.makedirs(chroma_persist_directory)
user_id = 'single_topic_rag_evaluation'
vector_store = MultiTenantVectorStore(chroma_persist_directory=chroma_persist_directory,
                                      embedding_model='text-embedding-3-small')

## 4. Index Documents

Index all document chunks into separate vector store collections:
- Each document gets its own collection (`document_0`, `document_1`, etc.)
- Allows isolated retrieval per document during evaluation
- Enables testing single-document question answering without cross-document contamination

In [ ]:
for index, splits in enumerate(all_splits):
    print('indexing document: ', index, 'number of splits: ', len(splits))
    vector_store.create_document_collection(user_id=user_id,
                                            collection_id=f'document_{index}',
                                            document_splits=splits,
                                            file_name=f'document_{index}')

## 5. Initialize Retrieval Chain

Create the `CitationRetrievalChain` for answer generation:
- Combines document retrieval with LLM-based answer generation
- **Note**: Retriever parameter is currently `None` (documents passed directly to invoke)

In [ ]:
# TODO: use vector store as retriever
retrieval_chain = CitationRetrievalChain(retriever=None,
                                         llm=create_llm())

## 6. Define Evaluation Function

The `retrieve_and_answer()` function orchestrates the complete evaluation pipeline:

### Process Flow
For each question in the dataset:
1. **Retrieve** top-k relevant document chunks from the vector store
2. **Generate** an answer using the retrieval chain
3. **Evaluate** the generated answer against the reference answer using LLM-as-judge
4. **Store** results including generated answer, referenced documents, score (0-5), and feedback


In [ ]:
def retrieve_and_answer(data):
    for index, question_row in data.iterrows():
        question = question_row['question']
        reference_answer = question_row.get('answer', "I don't know")
        document_index = question_row['document_index']
        collection_id = f'document_{document_index}'

        print(f'Processing question: {index}, collection_id: {collection_id}')

        found_documents = vector_store.retrieve_documents(user_id=user_id, collection_id=collection_id, query=question,
                                                          k=20)

        response = retrieval_chain.invoke(query=question, documents=found_documents, include_references_in_answer=False)

        generated_answer = response['answer']

        data.loc[index, 'generated_answer'] = generated_answer
        data.loc[index, 'referenced_documents'] = json.dumps(response['referenced_documents'])

        eval_result = evaluate_response(
            instruction=question,
            reference_answer=reference_answer,
            generated_answer=generated_answer
        )

        data.loc[index, 'score'] = eval_result.score
        data.loc[index, 'feedback'] = eval_result.feedback

    return data


## 7. Run Evaluation

Generate and evaluate answers for all three question types:

### Question Types
- **Single Passage**: Tests basic retrieval and comprehension
- **Multi Passage**: Tests ability to synthesize information across multiple chunks  
- **No Answer**: Tests hallucination prevention (system should respond "I don't know")

Results are saved to CSV files for later analysis.

In [ ]:
results_directory = '../../data/single_topic_rag_evaluation_dataset/results_3'
if not os.path.exists(results_directory):
    os.makedirs(results_directory)

In [ ]:
pd_documents.to_csv(f'{results_directory}/documents_predict.csv', index=False)

pd_single_passage = pd.read_csv(f'{input_dir}/single_passage_answer_questions.csv')
pd_single_passage_result = retrieve_and_answer(pd_single_passage)
pd_single_passage_result.to_csv(f'{results_directory}/single_passage_answer_questions_predict.csv', index=False)

pd_multi_passage = pd.read_csv(f'{input_dir}/multi_passage_answer_questions.csv')
pd_multi_passage_result = retrieve_and_answer(pd_multi_passage)
pd_multi_passage_result.to_csv(f'{results_directory}/multi_passage_answer_questions_predict.csv', index=False)

pd_no_answer = pd.read_csv(f'{input_dir}/no_answer_questions.csv')
pd_no_answer_result = retrieve_and_answer(pd_no_answer)
pd_no_answer_result.to_csv(f'{results_directory}/no_answer_questions_predict.csv', index=False)

## 8. Compare Results Across Runs

Analyze and visualize results from multiple evaluation runs to track performance changes.

### Metrics
- **Accuracy**: Normalized score (0-1 scale) = `sum(scores) / (num_questions * 5)`
- **Mean**: Average score across all questions
- **Median**: Median score

### Visualizations
Bar charts comparing accuracy across different runs for each question type.

In [ ]:
result_set = ["results", "results_2", "results_3"]


def summary(pd_result):
    total_questions = len(pd_result)
    score_sum = pd_result['score'].sum()
    accuracy = score_sum / (total_questions * 5)
    mean = pd_result['score'].mean()
    median = pd_result['score'].median()
    return {'accuracy': accuracy, 'mean': mean, 'median': median}


def print_config(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print("Evaluation Parameters:")
    print(yaml.dump(config, default_flow_style=False, sort_keys=False))


single_passage_accuracies = []
multi_passage_accuracies = []
no_answer_accuracies = []

for result in result_set:
    results_directory = f"../../data/single_topic_rag_evaluation_dataset/{result}"
    print(f'\nEvaluating {result}...')

    pd_single_passage_result = pd.read_csv(f'{results_directory}/single_passage_answer_questions_predict.csv')
    single_passage_summary = summary(pd_single_passage_result)

    pd_multi_passage_result = pd.read_csv(f'{results_directory}/multi_passage_answer_questions_predict.csv')
    multi_passage_summary = summary(pd_multi_passage_result)

    pd_no_answer_result = pd.read_csv(f'{results_directory}/no_answer_questions_predict.csv')
    no_answer_summary = summary(pd_no_answer_result)

    single_passage_accuracies.append(single_passage_summary['accuracy'])
    multi_passage_accuracies.append(multi_passage_summary['accuracy'])
    no_answer_accuracies.append(no_answer_summary['accuracy'])

    print(f"Single Passage Accuracy: {single_passage_summary['accuracy']:.2f}")
    print(f"Multi Passage Accuracy: {multi_passage_summary['accuracy']:.2f}")
    print(f"No Answer Accuracy: {no_answer_summary['accuracy']:.2f}")
    #print_config(f'{results_directory}/config.yaml')

x = np.arange(len(result_set))
color_palette = ['steelblue', 'darkorange', 'seagreen', 'crimson', 'goldenrod', 'purple']


def plot_accuracies(data, title):
    plt.figure()
    plt.bar(x, data, color=color_palette, tick_label=result_set)
    plt.ylim(0, 1)
    plt.ylabel('Accuracy')
    plt.title(title)
    plt.tight_layout()
    plt.show()


plot_accuracies(single_passage_accuracies, 'Single Passage Accuracy')
plot_accuracies(multi_passage_accuracies, 'Multi Passage Accuracy')
plot_accuracies(no_answer_accuracies, 'No Answer Accuracy')